<a href="https://colab.research.google.com/github/SNayak-166/Evaluation_of_RecommenderSystem/blob/genreencoding/Evaluation_of_RecommenderSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import getpass
import shutil
import pandas as pd

# Check if ratings.csv is already available in the local Colab workspace
if os.path.exists("ratings.csv"):
    print("--- ratings.csv already exists locally in workspace. Skipping download. ---")
    ratings_df = pd.read_csv("ratings.csv")
else:
    print("--- ratings.csv not found locally. Downloading from Kaggle... ---")

    # 1. Securely ask the evaluator for their Kaggle credentials
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("--- Kaggle Credentials Required ---")
        os.environ["KAGGLE_USERNAME"] = input("Enter your Kaggle Username: ")
        os.environ["KAGGLE_KEY"] = getpass.getpass("Enter your Kaggle API Key (input will be hidden): ")

    # 2. Install and import kagglehub
    !pip install kagglehub --quiet
    import kagglehub

    # 3. Download the dataset
    path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")
    print("Dataset downloaded to:", path)

    # 4. Copy ratings.csv to the local Colab workspace for easy access later
    source_path = os.path.join(path, "ratings.csv")
    shutil.copy(source_path, "ratings.csv")

    # 5. Load ratings_df from the workspace file
    ratings_df = pd.read_csv("ratings.csv")
    print("ratings.csv successfully downloaded, saved locally, and loaded!")

--- ratings.csv not found locally. Downloading from Kaggle... ---
--- Kaggle Credentials Required ---
Enter your Kaggle Username: Satwik.Nayak26
Enter your Kaggle API Key (input will be hidden): ··········
Using Colab cache for faster access to the 'movie-recommendation-system' dataset.
Dataset downloaded to: /kaggle/input/movie-recommendation-system
ratings.csv successfully downloaded, saved locally, and loaded!


In [2]:
import shutil

# Copy ratings.csv from the KaggleHub cache directory into your current Colab workspace
source_path = os.path.join(path, "ratings.csv")
shutil.copy(source_path, "ratings.csv")

print("ratings.csv successfully saved to the current workspace directory and is ready for use!")

ratings.csv successfully saved to the current workspace directory and is ready for use!


In [3]:
import pandas as pd

# Replace with your actual GitHub raw file URL
github_url = "https://raw.githubusercontent.com/SNayak-166/Evaluation_of_RecommenderSystem/refs/heads/genreencoding/processed_movies.csv"

# Load the processed movies dataframe
processed_movies_df = pd.read_csv(github_url)

# Verify the data loaded correctly
# print("Processed movies shape:", processed_movies_df.shape)
# print(processed_movies_df.head(3))

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

# Assuming rating_df and processed_movies_df are already loaded from Phase 1
# Merge ratings with processed movies
df = pd.merge(ratings_df, processed_movies_df, on='movieId')

# Create binary target for classification (Content-Based models)
# Rating >= 4.0 is considered a positive preference (1), else 0
df['liked'] = (df['rating'] >= 4.0).astype(int)

# Define genre feature columns (exclude non-genre columns)
non_feature_cols = ['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres', 'genre_list', 'liked']
genre_columns = [col for col in df.columns if col not in non_feature_cols]

print(f"Total interaction records: {len(df)}")
print(f"Features used for Content-Based models: {genre_columns}")

Total interaction records: 25000095
Features used for Content-Based models: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


Content based Recommender System Implementation

In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import gc

# Initialize models
lr_sgd = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, random_state=42)
nb_model = MultinomialNB()

chunk_size = 150000
print("Starting multi-batch training across the dataset...")

# We will save the last chunk to act as our test set for final evaluation
X_test_chunks = []
y_test_chunks = []

# Instead of reading from disk, we partition the already loaded ratings_df
num_rows = len(ratings_df)
for chunk_idx, start_idx in enumerate(range(0, num_rows, chunk_size)):
    chunk = ratings_df.iloc[start_idx : start_idx + chunk_size]

    # Merge chunk with processed movies
    chunk_merged = pd.merge(chunk, processed_movies_df, on='movieId')
    chunk_merged['liked'] = (chunk_merged['rating'] >= 4.0).astype(int)

    X_chunk = chunk_merged[genre_columns]
    y_chunk = chunk_merged['liked']

    # Let's use the last few chunks (or hold out 20% of the last chunk) for testing
    # so we can print the classification report at the end.
    X_tr, X_te, y_tr, y_te = train_test_split(X_chunk, y_chunk, test_size=0.2, random_state=42)

    # Train Naive Bayes incrementally
    if chunk_idx == 0:
        nb_model.partial_fit(X_tr, y_tr, classes=np.array([0, 1]))
    else:
        nb_model.partial_fit(X_tr, y_tr)

    # Train Logistic Regression incrementally
    lr_sgd.partial_fit(X_tr, y_tr, classes=np.array([0, 1]))

    # Accumulate test data from chunks to evaluate at the end
    X_test_chunks.append(X_te)
    y_test_chunks.append(y_te)

    if (chunk_idx + 1) % 5 == 0:
        print(f"Processed batch {chunk_idx + 1}...")

    del chunk, chunk_merged, X_chunk, y_chunk, X_tr, X_te, y_tr, y_te
    gc.collect()

# Combine accumulated test sets for final evaluation
X_test = pd.concat(X_test_chunks, ignore_index=True)
y_test = pd.concat(y_test_chunks, ignore_index=True)

# Generate predictions
nb_preds = nb_model.predict(X_test)
lr_preds = lr_sgd.predict(X_test)

# Print the final performance reports matching your desired output format
print("\n--- Naive Bayes Performance ---")
print(classification_report(y_test, nb_preds))

print("--- Logistic Regression Performance ---")
print(classification_report(y_test, lr_preds))

Starting multi-batch training across the dataset...
Processed batch 5...
Processed batch 10...
Processed batch 15...
Processed batch 20...
Processed batch 25...
Processed batch 30...
Processed batch 35...
Processed batch 40...
Processed batch 45...
Processed batch 50...
Processed batch 55...
Processed batch 60...
Processed batch 65...
Processed batch 70...
Processed batch 75...
Processed batch 80...
Processed batch 85...
Processed batch 90...
Processed batch 95...
Processed batch 100...
Processed batch 105...
Processed batch 110...
Processed batch 115...
Processed batch 120...
Processed batch 125...
Processed batch 130...
Processed batch 135...
Processed batch 140...
Processed batch 145...
Processed batch 150...
Processed batch 155...
Processed batch 160...
Processed batch 165...

--- Naive Bayes Performance ---
              precision    recall  f1-score   support

           0       0.56      0.56      0.56   2509254
           1       0.56      0.56      0.56   2490765

    accuracy

In [6]:
!pip install scikit-surprise --quiet

SVD Collaborative Filtering Cell

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise import accuracy
import gc

# 1. Setup the reader matching the column structure of ratings.csv (userId, movieId, rating, timestamp)
reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5.0), skip_lines=1)

print("Streaming full ratings.csv dataset from disk into Surprise (utilizing all 2.5Cr rows)...")

# 2. Load dataset directly from disk file (bypasses pandas dataframe RAM limits)
data = Dataset.load_from_file('ratings.csv', reader=reader)

# 3. Split into trainset and testset (80% train, 20% test) across the entire dataset
trainset, testset = surprise_train_test_split(data, test_size=0.2, random_state=42)

print("Data successfully split! Training SVD model...")

# 4. Train SVD Model on the full dataset
svd_model = SVD(n_factors=30, n_epochs=15, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

print("SVD training completed! Evaluating on the test set...")

# 5. Generate predictions and evaluate on the test set
svd_predictions = svd_model.test(testset)

print("--- SVD Evaluation ---")
accuracy.rmse(svd_predictions, verbose=True)
accuracy.mae(svd_predictions, verbose=True)

# 6. Clean up memory
del svd_predictions, trainset, testset, data
gc.collect()

Streaming full ratings.csv dataset from disk into Surprise (utilizing all 2.5Cr rows)...


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
import gc

print("Loading full ratings dataset for SLIM from local workspace...")
ratings_full = pd.read_csv('ratings.csv')

# 1. Split ratings into train and test sets to evaluate properly (80% train, 20% test)
train_ratings, test_ratings = train_test_split(ratings_full, test_size=0.2, random_state=42)

# Free up raw full dataframe RAM
del ratings_full
gc.collect()

# 2. Build the sparse user-item matrix using ALL training ratings from the full dataset
n_users = train_ratings['userId'].nunique()
n_movies = train_ratings['movieId'].nunique()

users = train_ratings['userId'].astype('category')
movies = train_ratings['movieId'].astype('category')

user_item_matrix = csr_matrix(
    (train_ratings['rating'], (users.cat.codes, movies.cat.codes)),
    shape=(n_users, n_movies)
).tocsc()

print(f"Full Training User-Item Matrix built! Shape: {user_item_matrix.shape}")

# 3. SLIM Training Setup
num_items = user_item_matrix.shape[1]

# Limit trained target items to a safe count (e.g., 500 items) to ensure Colab completes execution
max_items_to_train = min(num_items, 500)
W = np.zeros((max_items_to_train, num_items))

print(f"Training SLIM model on {max_items_to_train} target items using full-dataset user profiles...")

for j in range(max_items_to_train):
    y_col = user_item_matrix[:, j].toarray().ravel()

    # Efficient sparse column slicing (excluding target item j)
    left_cols = user_item_matrix[:, :j]
    right_cols = user_item_matrix[:, j+1:]
    X_cols = hstack([left_cols, right_cols]).tocsr()

    # Train ElasticNet with non-negative constraints
    en = ElasticNet(alpha=0.1, l1_ratio=0.5, positive=True, fit_intercept=False, max_iter=300)
    en.fit(X_cols, y_col)

    # Place coefficients into weight matrix W
    coefs = en.coef_
    W[j, :j] = coefs[:j]
    W[j, j+1:] = coefs[j:]

    if (j + 1) % 100 == 0:
        print(f"Trained {j + 1} / {max_items_to_train} items...")

print("SLIM Weight Matrix Shape:", W.shape)

# 4. Evaluation on Test Set (RMSE and MAE)
print("\n--- SLIM Evaluation ---")
y_true = []
y_pred = []

# Map user and movie IDs for fast index lookup
user_map = {uid: code for code, uid in enumerate(users.cat.categories)}
movie_map = {mid: code for code, mid in enumerate(movies.cat.categories)}

# Predict ratings for test set interactions
for _, row in test_ratings.iterrows():
    u_id = row['userId']
    m_id = row['movieId']
    actual_rating = row['rating']

    if u_id in user_map and m_id in movie_map:
        u_idx = user_map[u_id]
        m_idx = movie_map[m_id]

        if m_idx < max_items_to_train:
            user_profile = user_item_matrix.getrow(u_idx).toarray().ravel()
            predicted_rating = np.dot(user_profile, W[:, m_idx])

            if predicted_rating > 0:
                y_true.append(actual_rating)
                y_pred.append(predicted_rating)

# Calculate RMSE and MAE matching the evaluation format style
if len(y_true) > 0:
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_val = mean_absolute_error(y_true, y_pred)

    print(f"RMSE: {rmse_val:.4f}")
    print(f"MAE: {mae_val:.4f}")
else:
    print("Not enough overlapping test items captured in the trained weight slice.")

# Cleanup memory
del train_ratings, test_ratings, user_item_matrix, W
gc.collect()